In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!kaggle auth /content/drive/MyDrive/kaggle/access_token
!chmod 600 /content/drive/MyDrive/kaggle/access_token

!mkdir -p ~/.kaggle # creating .kaggle folder where the key should be placed
!cp /content/drive/MyDrive/kaggle/access_token ~/.kaggle/ # move the key to the folder

You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication


In [3]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p .
!unzip -q /content/chest-xray-pneumonia.zip

Dataset URL: https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia
License(s): other
100% 2.29G/2.29G [00:20<00:00, 122MB/s]



In [4]:
!git clone https://github.com/EdoardoGrassi/xai-binary-classification

Cloning into 'xai-binary-classification'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 25 (delta 5), reused 23 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 203.60 KiB | 1.35 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [5]:
# fix import paths into the repository
import sys
sys.path.insert(0,'./xai-binary-classification')

In [10]:
import os
from pathlib import Path

import numpy as np
import torch as tc
import torchvision.transforms.v2 as tvs
from torchvision.datasets import ImageFolder
from sklearn.metrics import accuracy_score, f1_score
from torch.utils.data import DataLoader
from tqdm import tqdm

from convnet import ConvNet
from datasets import ChestXRay
# from scatnet import ScatNet

RNG = tc.Generator().manual_seed(0)

transforms = tvs.Compose([
    tvs.Grayscale(),
    tvs.ToImage(),
    # tvs.CenterCrop(128),
    tvs.Resize((128, 128)),
    tvs.RandomHorizontalFlip(), # bit of data augmentation
    tvs.ConvertImageDtype(tc.float32),
])

ROOT = Path("/content/chest_xray")

dataset = tc.utils.data.ConcatDataset([
    ImageFolder(Path(ROOT, "train"), transform=transforms),
    ImageFolder(Path(ROOT, "val"), transform=transforms),
    # ImageFolder(Path(ROOT, "test"), transform=transforms),
])

BATCH_SIZE: int = 200
WORKERS: int = 2

train_dataset, valid_dataset = tc.utils.data.random_split(dataset, [0.8, 0.2], generator=RNG)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=True,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=WORKERS,
    pin_memory=True,
)

device = tc.device("cuda")

SHAPE = (1, 128, 128)
model = ConvNet(shape=SHAPE).to(device)
model.compile()

In [13]:
EPOCHS: int = 20

criterion = tc.nn.CrossEntropyLoss().to(device)
optimizer = tc.optim.Adam(model.parameters())

acc_train = []; f1_train = []; loss_train = []
acc_valid = []; f1_valid = []; loss_valid = []

# for epoch in tqdm(range(EPOCHS), "Training"):
for epoch in range(EPOCHS):
    print("Epoch:", epoch)

    running_loss = []
    running_acc = []
    running_f1 = []
    model.train()
    for images, labels in tqdm(train_loader, "Train on batches"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # Compute accuracy, and F1-score
        predicted = tc.argmax(outputs, dim=-1)
        running_loss.append(loss.item())
        running_acc.append(accuracy_score(labels.cpu(), predicted.cpu()))
        running_f1.append(f1_score(labels.cpu(), predicted.cpu(), average="weighted"))
        # print("Batch accuracy score", accuracy_score(labels.cpu(), predicted.cpu()))

    loss_train.append(np.mean(running_loss))
    acc_train.append(np.mean(running_acc))
    f1_train.append(np.mean(running_f1))

    print(
        f"  Train - Loss: {loss_train[epoch]:.4f}, Acc: {acc_train[epoch]:.4f}, F1: {f1_train[epoch]:.4f}"
    )

    # Validation phase
    model.eval()
    running_loss = []
    running_acc = []
    running_f1 = []
    with tc.no_grad():
        for images, labels in valid_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = tc.max(outputs, 1)
            running_loss.append(loss.item())
            running_acc.append(accuracy_score(labels.cpu(), predicted.cpu()))
            running_f1.append(
                f1_score(labels.cpu(), predicted.cpu(), average="weighted")
            )

        loss_valid.append(np.mean(running_loss))
        acc_valid.append(np.mean(running_acc))
        f1_valid.append(np.mean(running_f1))

        print(
            f"  Valid - Loss: {loss_valid[epoch]:.4f}, Acc: {acc_valid[epoch]:.4f}, F1: {f1_valid[epoch]:.4f}"
        )

Epoch: 0


Train on batches: 100%|██████████| 21/21 [00:52<00:00,  2.48s/it]

  Train - Loss: 0.4045, Acc: 0.9091, F1: 0.9065


  Valid - Loss: 0.1864, Acc: 0.9558, F1: 0.9562
Epoch: 1


Train on batches: 100%|██████████| 21/21 [00:54<00:00,  2.58s/it]

...

  Train - Loss: 0.0161, Acc: 0.9943, F1: 0.9943


  Valid - Loss: 0.0657, Acc: 0.9772, F1: 0.9769
Epoch: 19


Train on batches: 100%|██████████| 21/21 [00:52<00:00,  2.49s/it]

  Train - Loss: 0.0170, Acc: 0.9938, F1: 0.9938


  Valid - Loss: 0.1725, Acc: 0.9513, F1: 0.9496


In [15]:
# export the trained model

save_folder = Path("weights")
save_folder.mkdir(exist_ok=True)
tc.save(model.state_dict(), Path(save_folder, f"{model.name}.pt"))

In [20]:
# test the model

test_dataset = ImageFolder(Path(ROOT, "train"), transform=transforms)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=WORKERS,
    pin_memory=True,
)

def test_model(model: tc.nn.Module, images_loader: DataLoader):

    model.eval()
    all_preds = []
    all_labels = []

    with tc.no_grad():
        for images, labels in images_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            predicted = tc.argmax(outputs, dim=-1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    print(f'Test Accuracy: {accuracy:.4f}, Test F1-score: {f1:.4f}')

test_model(model, test_loader)

Test Accuracy: 0.9688, Test F1-score: 0.9681
